# 02 — Prophet Model
**Niloo**

In [1]:
import pandas as pd
import numpy as np
from prophet import Prophet
import joblib
import plotly.graph_objects as go
import os

In [3]:
df = pd.read_csv("/content/clean_data.csv")
df['ds'] = pd.to_datetime(df['ds'])

print("Shape:", df.shape)
print("Period:", df['ds'].min(), "->", df['ds'].max())
print(df.head())

Shape: (7428, 2)
Period: 1996-04-02 00:00:00 -> 2026-01-31 00:00:00
          ds     y
0 1996-04-02 -1.00
1 1996-04-03  2.70
2 1996-04-04  3.00
3 1996-04-05  2.10
4 1996-04-06  4.65


In [4]:
model = Prophet(interval_width=0.95, daily_seasonality=False)
model.fit(df)

In [5]:
future = model.make_future_dataframe(periods=1461, freq='D')
forecast = model.predict(future)

print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

             ds      yhat  yhat_lower  yhat_upper
8884 2030-01-27  1.918729   -2.927805    6.892938
8885 2030-01-28  1.885532   -3.132982    6.711379
8886 2030-01-29  1.860204   -3.209837    7.190012
8887 2030-01-30  1.848280   -3.315450    6.568446
8888 2030-01-31  1.815832   -3.072282    7.243579


In [7]:
fig = go.Figure()

# Historisk data
fig.add_trace(go.Scatter(
    x=df['ds'], y=df['y'],
    name='Historisk data',
    line=dict(color='#90A4AE', width=1)
))

# Konfidensintervall
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(0, 229, 255, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Konfidensintervall 95%'
))

# Prediktion
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    name='Prophet prediktion',
    line=dict(color='#00E5FF', width=2)
))

fig.update_layout(
    title='Temperaturprediktion 2026–2030 — nordcast',
    xaxis_title='Datum',
    yaxis_title='Temperatur (°C)',
    template='plotly_dark'
)

fig.show()

In [8]:
future_only = forecast[forecast['ds'] > '2026-01-31'].copy()
future_only['year'] = future_only['ds'].dt.year

summary = future_only.groupby('year')[['yhat', 'yhat_lower', 'yhat_upper']].mean().round(2)
print(summary)

       yhat  yhat_lower  yhat_upper
year                               
2026  10.69        5.66       15.69
2027   9.97        4.96       14.96
2028   9.97        4.93       14.97
2029  10.00        4.94       15.05
2030   2.07       -2.96        7.07


In [12]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/prophet_model.pkl")
print("Modell sparad!")

Modell sparad!


In [10]:
print(df['ds'].dt.year.value_counts().sort_index())

ds
1996    274
1997    334
2007    214
2008    366
2009    365
2010    365
2011    365
2012    366
2013    365
2014    365
2015    365
2016    366
2017    365
2018    365
2019    365
2020    366
2021    365
2022    365
2023    365
2024    366
2025    365
2026     31
Name: count, dtype: int64


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

historical = forecast[forecast['ds'] <= '2026-01-31'].copy()
historical = historical.merge(df, on='ds')

mae = mean_absolute_error(historical['y'], historical['yhat'])
rmse = np.sqrt(mean_squared_error(historical['y'], historical['yhat']))

print(f"MAE:  {mae:.2f}°C")
print(f"RMSE: {rmse:.2f}°C")

MAE:  2.00°C
RMSE: 2.57°C


In [13]:
from google.colab import files
files.download("../models/prophet_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import joblib
import os

os.makedirs("/content/models", exist_ok=True)
joblib.dump(model, "/content/models/prophet_model.pkl")
print("Storlek:", os.path.getsize("/content/models/prophet_model.pkl"), "bytes")

Storlek: 670017 bytes
